# Random Forest with Morgan Count Fingerprints — Chirality Baseline

This notebook establishes a baseline for understanding how molecular featurization affects a model's ability to predict chirality-related labels of increasing chemical difficulty.

We train a Random Forest on Morgan count fingerprints under **five featurization conditions**:

| Condition | Description | Deployable? |
|---|---|---|
| `morgan_count_chiral` | Full FP, `includeChirality=True` | ✓ |
| `morgan_count_no_chiral` | Full FP, `includeChirality=False` — **negative control** | ✓ |
| `morgan_delta_opp` | `FP_chiral(mol) − FP_chiral(enantiomer)` — antisymmetric, oracle | ✗ requires enantiomer at inference |
| `morgan_delta_nos` | `FP_chiral(mol) − FP_achiral(mol)` — deployable delta | ✓ |
| `morgan_cgr` | `[FP_achiral ‖ delta_nos]` — CGR-style concatenation (1024-dim) | ✓ |

Regarding deployment, obtaining the enantiomer SMILES at inference may be challenging. For this specific dataset, notebook 1 proved that it's trivial for these cases:
- Single stereocenter, known structure. Simply flip @ to @@ programmatically. This is what `get_enantiomer_smiles()` does and it requires nothing beyond the SMILES string.
- Multiple stereocenters: still algorithmically straightforward if you want the true enantiomer — flip all stereocenters simultaneously. Flipping only some gives a diastereomer, but the function `get_enantiomer_smiles()` handles this correctly for standard tetrahedral chirality.

However these scenarios could pose problems for obtaining the enatiomer
- Unassigned stereocenters: if a molecule's SMILES has no @/@@ annotation on a center that is structurally chiral (i.e., it was drawn without stereochemistry), `get_enantiomer_smiles()` cannot generate a meaningful enantiomer because the starting structure is already underspecified.
- Exotic stereo types (allenes, atropisomers, square planar): `get_enantiomer_smiles()` raises ValueError for these, which is the correct behavior. The real challenge is that generating the enantiomer of an atropisomer or axially chiral compound requires bond-level stereo inversion, not atom-level




We evaluate on three prediction targets of increasing chemical difficulty:

| Target | Description | Difficulty |
|---|---|---|
| `R/S_class` | CIP absolute configuration | Easiest — CIP label is directly incorporated into the Morgan atom invariants. |
| `@/@@_class` | SMILES parity tag | Harder — The `@/@@` SMILES token is formatting artifact. It's also near-independent of R/S and is not used by `includeChirality=True` at all, making it a formatting artifact the model must infer. |
| `F/L_class` | Chromatographic elution order | Hardest — 3D consequence of chirality. |

The results from this notebook show that **the standard `morgan_count_chiral` fingerprint is the most robust representation as it achieves the highest AUROC across all three prediction targets.**


## Key findings from notebooks 0 and 1
- `R/S` and `@/@@` are **nearly uncorrelated** (Cramér's V ≈ 0.02, κ ≈ 0.02). The `@/@@` tag depends on arbitrary SMILES atom ordering; CIP R/S is determined by atomic priority rules. They encode fundamentally different information.
- Predicting `@/@@` should be harder for Morgan FPs than predicting R/S, because the `@/@@` token is not explicitly encoded in the fingerprint (RDKit hashes the CIP R/S labels instead). The `@/@@` token is a SMILES formatting convention rather than a chemical property.
- The `F/L` elution order is the scientifically meaningful target.
- **Reference Paper Confirmation:** The reference paper correctly states that Morgan fingerprints specify the stereochemistry of chiral centers with CIP R/S labels. RDKit explicitly computes global 3D CIP priority rules rather than using the 1D syntactic `@/@@` markers.

## Note on the reference paper

The paper used this as a baseline:

```
Morgan fingerprints were generated with the extended connectivity fingerprints algorithm implemented by the RDKit library (2024.09.1). The GetMorganGenerator method of the rdFingerprintGenerator module was used for count fingerprints (with GetCountFingerprint). The fingerprints were generated with the following parameters: 
```
- radius = 3 (number of iterations to grow the fingerprint). this is the default.
- countSimulation = False. this is the default.

- **includeChirality = True (chirality information is added to the generated fingerprint). Default value is false.**

- useBondTypes = True (bond types are included as a part of the default bond invariants). this is the default.
- onlyNonzeroInvariants = False. this is the default.
- includeRingMembership = True. this is the default.
- countBounds = None (no boundaries for count simulation). this is the default.

- **fpSize = 512 (size of the generated fingerprint). Default is 2048**

- bondInvariantsGenerator = None. this is the default.
- atomInvariantsGenerator = None. this is the default.

In [1]:
import time

import numpy as np
import pandas as pd

from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    matthews_corrcoef,
    accuracy_score,
    f1_score,
    average_precision_score,
)

# Load the data and splits

In [2]:
# load stereoisomer data
# https://github.com/jairesdesousa/chiraldlsv/blob/main/TSNE_maps/class_all.csv
input_path = "data/class_all.csv"
df = pd.read_csv(input_path, index_col=0)
df

,SMILES,SMILES_opp,TR/TE,F/L_class,@/@@_class,R/S_class
0,Brc1ccc2c(c1)N[C@H](c1ccccc1)CC2,Brc1ccc2c(c1)N[C@@H](c1ccccc1)CC2,TE,F,@,S
1,Brc1ccc2c(c1)N[C@@H](c1ccccc1)CC2,Brc1ccc2c(c1)N[C@H](c1ccccc1)CC2,TE,L,@@,R
2,C#CCO[C@H](CSc1nc2cc(Cl)ccc2s1)CN(C)C(c1ccccc1...,C#CCO[C@@H](CSc1nc2cc(Cl)ccc2s1)CN(C)C(c1ccccc...,TE,F,@,S
3,C#CCO[C@@H](CSc1nc2cc(Cl)ccc2s1)CN(C)C(c1ccccc...,C#CCO[C@H](CSc1nc2cc(Cl)ccc2s1)CN(C)C(c1ccccc1...,TE,L,@@,R
4,C=C(C(C)=O)[C@@H](CC(=O)c1ccc(Br)cc1)C(=O)OCC,C=C(C(C)=O)[C@H](CC(=O)c1ccc(Br)cc1)C(=O)OCC,TE,F,@@,R
...,...,...,...,...,...,...
3853,OC[C@]1(CCCOCc2ccccc2)COCCO1,OC[C@@]1(CCCOCc2ccccc2)COCCO1,TR,L,@,R
3854,OCCCC[C@H](O)c1ccc(F)cc1,OCCCC[C@@H](O)c1ccc(F)cc1,TR,F,@,S
3855,OCCCC[C@@H](O)c1ccc(F)cc1,OCCCC[C@H](O)c1ccc(F)cc1,TR,L,@@,R
3856,OCCCC[C@]1(CO)COCCO1,OCCCC[C@@]1(CO)COCCO1,TR,L,@,S


In [3]:
df['TR/TE'].value_counts()

TR/TE
TR    3470
TE     388
Name: count, dtype: int64

In [4]:
df['F/L_class'].value_counts()

F/L_class
F    1929
L    1929
Name: count, dtype: int64

In [5]:
df['@/@@_class'].value_counts()

@/@@_class
@     1929
@@    1929
Name: count, dtype: int64

In [6]:
df['R/S_class'].value_counts()

R/S_class
S    1929
R    1929
Name: count, dtype: int64

In [7]:
def load_folds(path: str) -> list[dict[str, np.ndarray]]:
    """
    Load folds from a .npz file.

    Args:
        path: Path to .npz file saved by save_folds() in notebook 2.

    Returns:
        A list of dicts with keys 'train', 'val', and 'test', matching
        the output format of make_kfold_splits().
    """
    archive = np.load(path)
    fold_indices = sorted(set(
        int(k.split("_")[0].replace("fold", ""))
        for k in archive.files
    ))
    return [
        {
            "train": archive[f"fold{i}_train"],
            "val":   archive[f"fold{i}_val"],
            "test":  archive[f"fold{i}_test"],
        }
        for i in fold_indices
    ]

In [8]:
mol_folds = load_folds("data/cmrt_folds.npz")
print(f'Loaded {len(mol_folds)} folds')
for i, fold in enumerate(mol_folds):
    n_tr = len(fold['train'])
    n_v  = len(fold['val'])
    n_te = len(fold['test'])
    print(f'  Fold {i}: train={n_tr:,}  val={n_v:,}  test={n_te:,}')

Loaded 5 folds
  Fold 0: train=3,478  val=190  test=190
  Fold 1: train=3,478  val=190  test=190
  Fold 2: train=3,478  val=190  test=190
  Fold 3: train=3,478  val=190  test=190
  Fold 4: train=3,478  val=190  test=190


# Fingerprint helper functions

In [9]:
def rdkit_to_np(vect, num_bits: int) -> np.ndarray:
    """Convert a sparse RDKit fingerprint vector to a dense float64 numpy array.

    Using float64 (rather than the default uint32 from GetCountFingerprintAsNumPy)
    prevents integer overflow when fingerprints are subtracted to form delta vectors.

    Args:
        vect: An RDKit ExplicitBitVect or UIntSparseIntVect fingerprint.
        num_bits: Length of the fingerprint (must match fpSize used to generate vect).

    Returns:
        Dense 1D numpy array of shape (num_bits,) and dtype float64.
    """
    arr = np.zeros((num_bits,), dtype=np.float64)
    DataStructs.ConvertToNumpyArray(vect, arr)  # overwrites arr in-place
    return arr


def calc_morgan_fp(
    smi: str,
    count: bool = True,
    radius: int = 3,
    fpSize: int = 512,
    includeChirality: bool = True,
    useFeatures: bool = False,
) -> np.ndarray:
    """Compute a Morgan (ECFP-style) count fingerprint for a SMILES string.

    Args:
        smi: SMILES string of the molecule.
        count: If True, returns a count fingerprint; if False, a bit fingerprint.
        radius: Number of bond hops to consider per atom environment.
        fpSize: Length of the output fingerprint vector.
        includeChirality: If True, the CIP R/S label is incorporated into
            atom invariants at each radius. Setting False makes enantiomers
            produce identical vectors.
        useFeatures: If True, uses pharmacophoric atom invariants (FCFP-style).

    Returns:
        Dense 1D numpy array of shape (fpSize,) and dtype float64.
    """
    mol = Chem.MolFromSmiles(smi)
    atom_inv_gen = (
        rdFingerprintGenerator.GetMorganFeatureAtomInvGen() if useFeatures else None
    )
    morgan_gen = rdFingerprintGenerator.GetMorganGenerator(
        radius=radius,
        fpSize=fpSize,
        includeChirality=includeChirality,
        atomInvariantsGenerator=atom_inv_gen,
    )
    fp = getattr(morgan_gen, f'Get{"Count" if count else ""}Fingerprint')(mol)
    return rdkit_to_np(fp, fpSize)

# Fingerprint inspection: one example pair

Before computing features for the full dataset, we inspect a single enantiomeric pair
to build intuition about what each featurization condition actually encodes.

**Test pair (rows 0 and 1):**
- Row 0: `Brc1ccc2c(c1)N[C@H](c1ccccc1)CC2` — CIP label: S
- Row 1: `Brc1ccc2c(c1)N[C@@H](c1ccccc1)CC2` — CIP label: R

Key questions to answer:
1. How many fingerprint bits differ between the chiral and achiral version of the same molecule?
2. How many bits differ between the two enantiomers?
3. Is the set of bits that change due to chirality (`delta_nos`) the same set that differs between enantiomers (`delta_opp`)?
4. Is `delta_opp` exactly antisymmetric (i.e., does `delta(A) = -delta(B)`)?"

In [10]:
# ── Fingerprint parameters (paper defaults) ───────────────────────────────────
FP_PARAMS = {"count": True, "radius": 3, "fpSize": 512, "useFeatures": False}

# ── Example pair ──────────────────────────────────────────────────────────────
smi_A   = df.iloc[0]["SMILES"]      # @  -> S
smi_B   = df.iloc[1]["SMILES"]      # @@ -> R
smi_opp = df.iloc[0]["SMILES_opp"]  # enantiomer of row 0 (should equal row 1 SMILES)
assert smi_opp == smi_B, "SMILES_opp of row 0 should equal SMILES of row 1"

print(f"Molecule A (row 0, @, S):  {smi_A}")
print(f"Molecule B (row 1, @@, R): {smi_B}")
print()

# ── Compute the four base fingerprints ───────────────────────────────────────
fp_A_chiral  = calc_morgan_fp(smi_A, includeChirality=True,  **FP_PARAMS)
fp_B_chiral  = calc_morgan_fp(smi_B, includeChirality=True,  **FP_PARAMS)
fp_A_achiral = calc_morgan_fp(smi_A, includeChirality=False, **FP_PARAMS)
fp_B_achiral = calc_morgan_fp(smi_B, includeChirality=False, **FP_PARAMS)

print("Are achiral FPs of A and B identical? (they must be — same connectivity):",
      np.array_equal(fp_A_achiral, fp_B_achiral))
print()

# ── Bit set analysis ──────────────────────────────────────────────────────────
bits_A_chiral  = set(np.where(fp_A_chiral  != 0)[0])
bits_B_chiral  = set(np.where(fp_B_chiral  != 0)[0])
bits_achiral   = set(np.where(fp_A_achiral != 0)[0])

shared_chiral_achiral = bits_A_chiral & bits_achiral
only_in_chiral        = bits_A_chiral - bits_achiral
only_in_achiral       = bits_achiral  - bits_A_chiral

print("── Bit overlap: chiral A vs achiral A ───────────────────────────────────")
print(f"  Bits active in chiral A:          {len(bits_A_chiral):>4}")
print(f"  Bits active in achiral A:         {len(bits_achiral):>4}")
print(f"  Shared bits (in both):            {len(shared_chiral_achiral):>4}")
print(f"  Bits only in chiral (new from @): {len(only_in_chiral):>4}")
print(f"  Bits only in achiral (lost by @): {len(only_in_achiral):>4}")
print()

print("── Bit overlap: chiral A vs chiral B (the two enantiomers) ─────────────")
shared_enantiomers   = bits_A_chiral & bits_B_chiral
only_in_A            = bits_A_chiral - bits_B_chiral
only_in_B            = bits_B_chiral - bits_A_chiral
print(f"  Bits active in chiral A:          {len(bits_A_chiral):>4}")
print(f"  Bits active in chiral B:          {len(bits_B_chiral):>4}")
print(f"  Shared bits (same in both):       {len(shared_enantiomers):>4}")
print(f"  Bits only in A (lost by flip):    {len(only_in_A):>4}")
print(f"  Bits only in B (gained by flip):  {len(only_in_B):>4}")

Molecule A (row 0, @, S):  Brc1ccc2c(c1)N[C@H](c1ccccc1)CC2
Molecule B (row 1, @@, R): Brc1ccc2c(c1)N[C@@H](c1ccccc1)CC2

Are achiral FPs of A and B identical? (they must be — same connectivity): True

── Bit overlap: chiral A vs achiral A ───────────────────────────────────
  Bits active in chiral A:            44
  Bits active in achiral A:           43
  Shared bits (in both):              32
  Bits only in chiral (new from @):   12
  Bits only in achiral (lost by @):   11

── Bit overlap: chiral A vs chiral B (the two enantiomers) ─────────────
  Bits active in chiral A:            44
  Bits active in chiral B:            44
  Shared bits (same in both):         32
  Bits only in A (lost by flip):      12
  Bits only in B (gained by flip):    12


In [11]:
# ── Delta vector analysis ─────────────────────────────────────────────────────
delta_opp_A = fp_A_chiral - fp_B_chiral   # ori - enantiomer (for molecule A)
delta_opp_B = fp_B_chiral - fp_A_chiral   # ori - enantiomer (for molecule B)
delta_nos_A = fp_A_chiral - fp_A_achiral  # ori - no_stereo  (for molecule A)
delta_nos_B = fp_B_chiral - fp_B_achiral  # ori - no_stereo  (for molecule B)

print("── Delta vector sparsity ────────────────────────────────────────────────")
print(f"  delta_opp_A nonzero bits: {(delta_opp_A != 0).sum():>4}  range: [{delta_opp_A.min():.0f}, {delta_opp_A.max():.0f}]")
print(f"  delta_opp_B nonzero bits: {(delta_opp_B != 0).sum():>4}  range: [{delta_opp_B.min():.0f}, {delta_opp_B.max():.0f}]")
print(f"  delta_nos_A nonzero bits: {(delta_nos_A != 0).sum():>4}  range: [{delta_nos_A.min():.0f}, {delta_nos_A.max():.0f}]")
print(f"  delta_nos_B nonzero bits: {(delta_nos_B != 0).sum():>4}  range: [{delta_nos_B.min():.0f}, {delta_nos_B.max():.0f}]")
print()

# ── Antisymmetry check ────────────────────────────────────────────────────────
print("── Antisymmetry check (delta_opp) ───────────────────────────────────────")
is_antisymmetric = np.allclose(delta_opp_A, -delta_opp_B)
print(f"  delta_opp_A == -delta_opp_B: {is_antisymmetric}")
print(f"  Interpretation: {'✓ antisymmetric — model receives opposite signals for enantiomers' if is_antisymmetric else '✗ unexpected'}")
print()

# ── Is delta_nos antisymmetric? ───────────────────────────────────────────────
print("── Antisymmetry check (delta_nos) ───────────────────────────────────────")
is_anti_nos = np.allclose(delta_nos_A, -delta_nos_B)
are_equal   = np.allclose(delta_nos_A,  delta_nos_B)
print(f"  delta_nos_A == -delta_nos_B: {is_anti_nos}")
print(f"  delta_nos_A ==  delta_nos_B: {are_equal}")
print(f"  Interpretation: delta_nos is neither antisymmetric nor identical")
print(f"  — both enantiomers have a unique delta relative to the shared achiral scaffold")
print()

# ── Do delta_nos and delta_opp flag the same bits? ───────────────────────────
print("── Bit-position overlap: delta_opp_A vs delta_nos_A ────────────────────")
nonzero_opp = set(np.where(delta_opp_A != 0)[0])
nonzero_nos = set(np.where(delta_nos_A != 0)[0])
print(f"  Nonzero positions in delta_opp_A: {len(nonzero_opp)}")
print(f"  Nonzero positions in delta_nos_A: {len(nonzero_nos)}")
print(f"  Shared nonzero positions:         {len(nonzero_opp & nonzero_nos)}")
print(f"  Only in delta_opp:                {len(nonzero_opp - nonzero_nos)}")
print(f"  Only in delta_nos:                {len(nonzero_nos - nonzero_opp)}")
print()

# ── Reconstructability check ─────────────────────────────────────────────────
print("── Reconstructability: achiral + delta_nos == chiral? ───────────────────")
reconstructed_A = fp_A_achiral + delta_nos_A
reconstructed_B = fp_B_achiral + delta_nos_B
print(f"  A: fp_achiral + delta_nos_A == fp_A_chiral: {np.allclose(reconstructed_A, fp_A_chiral)}")
print(f"  B: fp_achiral + delta_nos_B == fp_B_chiral: {np.allclose(reconstructed_B, fp_B_chiral)}")
print()
print("  Implication: [fp_achiral | delta_nos] (CGR-style) contains identical")
print("  information to fp_chiral — no information gain, but explicit factorization")
print("  of connectivity vs. chirality signals may help tree-based models.")

── Delta vector sparsity ────────────────────────────────────────────────
  delta_opp_A nonzero bits:   24  range: [-1, 1]
  delta_opp_B nonzero bits:   24  range: [-1, 1]
  delta_nos_A nonzero bits:   24  range: [-1, 1]
  delta_nos_B nonzero bits:   24  range: [-1, 1]

── Antisymmetry check (delta_opp) ───────────────────────────────────────
  delta_opp_A == -delta_opp_B: True
  Interpretation: ✓ antisymmetric — model receives opposite signals for enantiomers

── Antisymmetry check (delta_nos) ───────────────────────────────────────
  delta_nos_A == -delta_nos_B: False
  delta_nos_A ==  delta_nos_B: False
  Interpretation: delta_nos is neither antisymmetric nor identical
  — both enantiomers have a unique delta relative to the shared achiral scaffold

── Bit-position overlap: delta_opp_A vs delta_nos_A ────────────────────
  Nonzero positions in delta_opp_A: 24
  Nonzero positions in delta_nos_A: 24
  Shared nonzero positions:         12
  Only in delta_opp:                12
  Only i

# Define target variables

All three targets are binary and perfectly balanced (50/50) by dataset construction — every
molecule appears alongside its enantiomer, so every class has an equal counterpart.

**Chance baseline for all three targets: 50% accuracy.**

In [12]:
TARGET_LABEL_MAPS = {
    "R/S_class":  {"R": 0, "S": 1},
    "@/@@_class": {"@": 0, "@@": 1},
    "F/L_class":  {"F": 0, "L": 1},
}

for col, label_map in TARGET_LABEL_MAPS.items():
    df[f"label_{col}"] = df[col].map(label_map)

print("Class distributions (all should be 50/50):")
for col in TARGET_LABEL_MAPS:
    vc = df[f"label_{col}"].value_counts(normalize=True)
    print(f"  {col}: {vc.to_dict()}")

Class distributions (all should be 50/50):
  R/S_class: {1: 0.5, 0: 0.5}
  @/@@_class: {0: 0.5, 1: 0.5}
  F/L_class: {0: 0.5, 1: 0.5}


# Obtaining the Enantiomer SMILES

For this dataset `SMILES_opp` is provided directly, making `X_opp` trivial to compute. In a general deployment setting, the enantiomer SMILES can be generated programmatically via `get_enantiomer_smiles()` from notebook 1 for standard tetrahedral stereocenters; however the concern is that a model trained on antisymmetric paired features may not generalize straightforwardly to single unpaired query molecules.

Because of this distributional mismatch between training (paired data) and inference (unpaired query), the `morgan_delta_nos` and `morgan_cgr` representations are preferred for real-world deployment.

# Precompute all fingerprints

All five featurization conditions are derived from three base fingerprint matrices.
We compute these once over the entire dataset before the fold loop for two reasons:

1. **Efficiency**: avoids redundant featurization on the same molecules across folds and conditions.
2. **Correctness**: delta conditions require row-aligned subtraction across all 3858 molecules;
   computing per-fold would require careful re-alignment.

The fold loop then slices these precomputed matrices by index.

### The five conditions

**`morgan_count_chiral`** — the standard chiral Morgan fingerprint. Chirality is encoded by
incorporating the CIP R/S label into atom invariants at each radius. Enantiomers
produce different vectors; the model must discover which differences matter.

**`morgan_count_no_chiral`** — negative control. Setting `includeChirality=False` makes
enantiomers produce identical vectors. Any model trained on this should score at chance (50%).

**`morgan_delta_opp`** — `FP_chiral(mol) − FP_chiral(enantiomer)`. This is the
paper's "ori-opp" descriptor. It is **antisymmetric** by construction: the signal for
molecule A is exactly the negative of the signal for molecule B. This makes chirality
maximally explicit and should produce near-perfect `@/@@` and `R/S` performance.
⚠️ **Not preferred for deployment**: If you train a Random Forest exclusively on delta_opp, the model doesn't fully understand the identity of the molecule. It doesn't know if it's looking at a tiny ibuprofen derivative or a massive macrocyclic antibiotic because the featurization only captures the isolated stereocenter differences. Look at the math of what happens when you subtract an enantiomer's fingerprint from its counterpart:
- FP(query) = `[Achiral Scaffold Bits] + [Chiral Bits A]`
- FP(enantiomer) = `[Achiral Scaffold Bits] + [Chiral Bits B]`
- morgan_delta_opp = `[Chiral Bits A] - [Chiral Bits B]`
- The entire achiral scaffold cancels out to zero.

**`morgan_delta_nos`** — `FP_chiral(mol) − FP_achiral(mol)`. The paper's "ori-ns"
descriptor. Captures which fingerprint bits appear or disappear when stereo annotations
are added to the achiral scaffold. Deployable from a single molecule. However, this featurization also removes the common achiral scaffold similar to `morgan_delta_opp`. However `morgan_delta_nos` is not antisymmetric:
- `delta_opp` forces the model to learn the difference between two enantiomers (which cancels the scaffold)
- `delta_nos` forces the model to learn the "signature" of chirality relative to a neutral background. But the feature vector has still stripped away the scaffold identity which forces the model to make a prediction based solely on the "chiral signature" bits.

**`morgan_cgr`** — `[FP_achiral ‖ (FP_chiral − FP_achiral)]`. CGR (Condensed Graph of Reaction)-style concatenation. Part 1 encodes connectivity; part 2 encodes the chirality perturbation. Mathematically equivalent to `FP_chiral` (since achiral + delta = chiral),
but the explicit factorization may help tree-based models find chirality-relevant splits without needing to rediscover the factorization from the interleaved hash space.
Doubles the feature dimensionality to 1024.

In [13]:
print("Precomputing fingerprints for all molecules...")
START = time.time()

X_chiral  = np.stack(df["SMILES"].apply(
    calc_morgan_fp, includeChirality=True,  **FP_PARAMS).values)
X_achiral = np.stack(df["SMILES"].apply(
    calc_morgan_fp, includeChirality=False, **FP_PARAMS).values)
X_opp     = np.stack(df["SMILES_opp"].apply(
    calc_morgan_fp, includeChirality=True,  **FP_PARAMS).values)

print(f"  Base FP matrices computed in {time.time() - START:.1f}s")
print(f"  X_chiral:  {X_chiral.shape}")
print(f"  X_achiral: {X_achiral.shape}")
print(f"  X_opp:     {X_opp.shape}")

# Sanity check: achiral FPs of paired enantiomers must be identical
# (same connectivity, only stereo differs). Check on all pairs.
assert np.array_equal(X_achiral, np.stack(df["SMILES_opp"].apply(
    calc_morgan_fp, includeChirality=False, **FP_PARAMS).values)), \
    "Achiral FPs of SMILES and SMILES_opp differ — unexpected"
print("  ✓ Achiral FPs are identical for all enantiomeric pairs")

Precomputing fingerprints for all molecules...
  Base FP matrices computed in 1.3s
  X_chiral:  (3858, 512)
  X_achiral: (3858, 512)
  X_opp:     (3858, 512)
  ✓ Achiral FPs are identical for all enantiomeric pairs


In [14]:
# ── Compute delta matrices ─────────────────────────────────────────────────────
X_delta_opp = X_chiral - X_opp      # antisymmetric; requires enantiomer at inference
X_delta_nos = X_chiral - X_achiral  # deployable; does not require enantiomer
X_cgr       = np.concatenate([X_achiral, X_delta_nos], axis=1)  # 1024-dim

print("Delta and CGR matrices:")
print(f"  X_delta_opp: {X_delta_opp.shape}  nonzero per row: {(X_delta_opp != 0).sum(axis=1).mean():.1f} avg")
print(f"  X_delta_nos: {X_delta_nos.shape}  nonzero per row: {(X_delta_nos != 0).sum(axis=1).mean():.1f} avg")
print(f"  X_cgr:       {X_cgr.shape}        (= achiral 512 + delta_nos 512)")
print()

# Verify antisymmetry of delta_opp across all pairs
# For each even index i (molecule A), its enantiomer is at i+1 (molecule B)
# delta_opp[i] should equal -delta_opp[i+1]
even_idx = np.arange(0, len(df), 2)
odd_idx  = even_idx + 1
antisymmetry_check = np.allclose(X_delta_opp[even_idx], -X_delta_opp[odd_idx])
print(f"✓ delta_opp is antisymmetric across all {len(even_idx)} pairs: {antisymmetry_check}")

# Verify CGR reconstructs chiral FP
cgr_nos_part = X_cgr[:, 512:]   # second half of CGR
cgr_ach_part = X_cgr[:, :512]   # first half of CGR
reconstructed = cgr_ach_part + cgr_nos_part
print(f"✓ achiral + delta_nos reconstructs chiral FP exactly: {np.allclose(reconstructed, X_chiral)}")

Delta and CGR matrices:
  X_delta_opp: (3858, 512)  nonzero per row: 15.2 avg
  X_delta_nos: (3858, 512)  nonzero per row: 17.0 avg
  X_cgr:       (3858, 1024)        (= achiral 512 + delta_nos 512)

✓ delta_opp is antisymmetric across all 1929 pairs: True
✓ achiral + delta_nos reconstructs chiral FP exactly: True


# Featurization conditions and hyperparameters

`FEATURIZATION_CONDITIONS` maps condition name → precomputed numpy array.
The training loop slices by fold index rather than calling the fingerprint function.

In [15]:
FEATURIZATION_CONDITIONS = {
    "morgan_count_chiral":    X_chiral,     # 512-dim, positive condition
    "morgan_count_no_chiral": X_achiral,    # 512-dim, negative control
    "morgan_delta_opp":       X_delta_opp,  # 512-dim, antisymmetric oracle (not deployable)
    "morgan_delta_nos":       X_delta_nos,  # 512-dim, deployable delta
    "morgan_cgr":             X_cgr,        # 1024-dim, CGR-style concatenation
}

RF_HYPERPARAMETERS = {
    "n_estimators": 100,
    "max_depth": 25,
    "random_state": 42,
}

TARGETS = list(TARGET_LABEL_MAPS.keys())  # ['R/S_class', '@/@@_class', 'F/L_class']
print(f"Conditions: {list(FEATURIZATION_CONDITIONS.keys())}")
print(f"Targets:    {TARGETS}")
print(f"Folds:      {len(mol_folds)}")
print(f"Total RF fits: {len(FEATURIZATION_CONDITIONS)} × {len(TARGETS)} × {len(mol_folds)} = "      f"{len(FEATURIZATION_CONDITIONS) * len(TARGETS) * len(mol_folds)}")

Conditions: ['morgan_count_chiral', 'morgan_count_no_chiral', 'morgan_delta_opp', 'morgan_delta_nos', 'morgan_cgr']
Targets:    ['R/S_class', '@/@@_class', 'F/L_class']
Folds:      5
Total RF fits: 5 × 3 × 5 = 75


# Evaluation metrics

### Metrics that require **predicted probabilities**

**AUROC** — measures rank ordering of positives above negatives across all thresholds.
Range [0, 1]; 0.5 = chance, 1.0 = perfect. Threshold-independent; the standard
comparison metric across classifiers.

**AUPRC** — precision-recall tradeoff across thresholds. Most informative at class
imbalance, but still a useful complement at 50/50 balance. Baseline ≈ 0.5.

### Metrics that require **hard class predictions** (threshold = 0.5)

**Accuracy** — `(TP + TN) / total`. Valid and interpretable here because classes are
exactly balanced (50% baseline). Misleading at imbalance.

**F1 Score** — harmonic mean of precision and recall. Close to accuracy at 50/50 balance
but more informative at imbalance. Requires discrete TP/FP/FN counts.

**MCC (Matthews Correlation Coefficient)** — range [-1, 1], 0 = chance. Uses all four
confusion matrix cells; considered the single most informative binary classification
metric (Chicco & Jurman 2020, Pat Walters). Symmetric across both classes."

In [16]:
def evaluate(
    model: RandomForestClassifier,
    X_test: np.ndarray,
    y_test: np.ndarray,
) -> dict:
    """Compute all evaluation metrics for a single fold's test set.

    Args:
        model: A fitted RandomForestClassifier.
        X_test: Feature matrix for the test set, shape (n_samples, n_features).
        y_test: True binary labels for the test set, shape (n_samples,).

    Returns:
        Dict mapping metric name to scalar value.
    """
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    y_pred       = model.predict(X_test)
    return {
        "AUROC":    roc_auc_score(y_test, y_pred_proba),
        "MCC":      matthews_corrcoef(y_test, y_pred),
        "Accuracy": accuracy_score(y_test, y_pred),
        "F1":       f1_score(y_test, y_pred),
        "AUPRC":    average_precision_score(y_test, y_pred_proba),
    }

# Training and evaluation

Outer loop: featurization condition (5 conditions)
Middle loop: prediction target (3 targets: R/S, @/@@, F/L)
Inner loop: fold (5 independent train/test splits)

**Total: 75 RF fits.**

**Note on the validation set:** Random forests have no early stopping, so the `val` split
is not used for model fitting. However, it is now scored after each fit and stored in
`all_results` for records alongside train and test metrics. Only test set scores
will be used for the Tukey HSD analysis.

**Note on `morgan_delta_opp`:** This condition requires access to the paired enantiomer
at inference time and is therefore not deployable. It serves as an oracle / upper bound
to show the maximum performance achievable when the antisymmetric chirality signal is
made fully explicit."

In [17]:
all_results = [] # train + val + test; saved to CSV for records

for condition_name, X_all in FEATURIZATION_CONDITIONS.items():
    print(f"\n{'#' * 70}")
    print(f"Condition: {condition_name}  (feature dim={X_all.shape[1]})")
    if condition_name == "morgan_delta_opp":
        print(f"  ⚠ Oracle condition — requires enantiomer at inference; not deployable")
    print(f"{'#' * 70}")

    for target_col in TARGETS:
        label_col = f"label_{target_col}"
        print(f"\n  Target: {target_col}")
        print(f"  {'-' * 60}")

        for fold_idx, fold in enumerate(mol_folds):
            START = time.time()

            # Slice precomputed feature matrices and labels by fold index
            # val only used as another test set. Tukey HSD will only be calculated on test set
            X_train = X_all[fold["train"]]
            X_val   = X_all[fold["val"]]
            X_test  = X_all[fold["test"]]
            y_train = df[label_col].iloc[fold["train"]].values
            y_val   = df[label_col].iloc[fold["val"]].values
            y_test  = df[label_col].iloc[fold["test"]].values

            model = RandomForestClassifier(**RF_HYPERPARAMETERS)
            model.fit(X_train, y_train)

            elapsed = time.time() - START

            # Evaluate on all three splits
            test_metrics  = evaluate(model, X_test,  y_test)
            train_metrics = evaluate(model, X_train, y_train)
            val_metrics   = evaluate(model, X_val,   y_val)

            # Tag each dict with fold metadata
            for metrics, split in [
                (test_metrics,  "test"),
                (train_metrics, "train"),
                (val_metrics,   "val"),
            ]:
                metrics["fold"]          = fold_idx
                metrics["model"]         = "RF"
                metrics["featurization"] = condition_name
                metrics["target"]        = target_col
                metrics["split"]         = split

            # full list captures all three splits for records
            all_results.extend([train_metrics, val_metrics, test_metrics])

            # Print only test-set metrics inline; train/val AUROC shown compactly
            print(
                f"  fold {fold_idx} | "
                f"test  AUROC={test_metrics['AUROC']:.3f}  "
                f"MCC={test_metrics['MCC']:.3f}  "
                f"Acc={test_metrics['Accuracy']:.3f}  "
                f"F1={test_metrics['F1']:.3f}  "
                f"AUPRC={test_metrics['AUPRC']:.3f}  "
                f"({elapsed:.1f}s)\n"
                f"         | "
                f"train AUROC={train_metrics['AUROC']:.3f}  "
                f"val   AUROC={val_metrics['AUROC']:.3f}"
            )


######################################################################
Condition: morgan_count_chiral  (feature dim=512)
######################################################################

  Target: R/S_class
  ------------------------------------------------------------
  fold 0 | test  AUROC=0.940  MCC=0.675  Acc=0.837  F1=0.841  AUPRC=0.942  (0.6s)
         | train AUROC=0.994  val   AUROC=0.918
  fold 1 | test  AUROC=0.939  MCC=0.705  Acc=0.853  F1=0.853  AUPRC=0.946  (0.6s)
         | train AUROC=0.994  val   AUROC=0.947
  fold 2 | test  AUROC=0.950  MCC=0.717  Acc=0.858  F1=0.854  AUPRC=0.950  (0.6s)
         | train AUROC=0.994  val   AUROC=0.935
  fold 3 | test  AUROC=0.934  MCC=0.663  Acc=0.832  F1=0.832  AUPRC=0.938  (0.6s)
         | train AUROC=0.994  val   AUROC=0.924
  fold 4 | test  AUROC=0.958  MCC=0.705  Acc=0.853  F1=0.853  AUPRC=0.961  (0.6s)
         | train AUROC=0.994  val   AUROC=0.942

  Target: @/@@_class
  -------------------------------------------------

# Summary across folds

Mean ± std across the 5 independent test sets. These 5 per-fold scores per condition
are what will be used in the Tukey HSD test in the final analysis notebook."

In [18]:
all_results_df = pd.DataFrame(all_results)  # train + val + test (225 rows)
test_results_df = all_results_df[all_results_df["split"] == "test"]  # test only (75 rows)
metric_cols = ["AUROC", "MCC", "Accuracy", "F1", "AUPRC"]

summary = (
    test_results_df
    .groupby(["featurization", "target"])[metric_cols]
    .agg(["mean", "std"])
    .round(4)
)

target_order    = ["R/S_class", "@/@@_class", "F/L_class"]
condition_order = list(FEATURIZATION_CONDITIONS.keys())  # preserve experimental ordering
summary = summary.reindex([(c, t) for c in condition_order for t in target_order])

print("Test Set Performance — Mean ± std across 5 folds:")
print(summary.to_string())

Test Set Performance — Mean ± std across 5 folds:
                                    AUROC             MCC         Accuracy              F1           AUPRC        
                                     mean     std    mean     std     mean     std    mean     std    mean     std
featurization          target                                                                                     
morgan_count_chiral    R/S_class   0.9442  0.0097  0.6930  0.0229   0.8463  0.0114  0.8464  0.0098  0.9473  0.0087
                       @/@@_class  0.8029  0.0338  0.4200  0.0720   0.7095  0.0364  0.7180  0.0326  0.8107  0.0405
                       F/L_class   0.8637  0.0140  0.5307  0.0368   0.7653  0.0185  0.7643  0.0218  0.8612  0.0142
morgan_count_no_chiral R/S_class   0.5000  0.0000  0.0000  0.0000   0.5000  0.0000  0.4876  0.0261  0.5000  0.0000
                       @/@@_class  0.5000  0.0000  0.0000  0.0000   0.5000  0.0000  0.4970  0.0298  0.5000  0.0000
                       F/L_cla

# Sanity checks

**Negative control** (`morgan_count_no_chiral`): all targets should be at chance (≈50%).

**Delta conditions**: `morgan_delta_opp` (antisymmetric oracle) should outperform
`morgan_delta_nos` (deployable). Both should outperform `morgan_count_chiral` on
`@/@@` and `R/S` where the delta signal is most explicit. For `F/L`, the advantage
may be smaller since elution order reflects 3D structure beyond what the 2D delta captures.

**CGR condition**: performance should be close to `morgan_count_chiral` since they
contain identical information. Any gap reflects whether the explicit factorization
helps the RF find chirality-relevant splits."

In [19]:
print(f"{'Target':<15} {'Condition':<28} {'AUROC':>7} {'MCC':>7} {'Accuracy':>9} {'Check'}")
print("-" * 80)
for target in target_order:
    for condition in condition_order:
        subset = test_results_df[
            (test_results_df["target"] == target) &
            (test_results_df["featurization"] == condition)
        ]
        auroc = subset["AUROC"].mean()
        mcc   = subset["MCC"].mean()
        acc   = subset["Accuracy"].mean()

        flag = ""
        if condition == "morgan_count_no_chiral":
            flag = "✓ at chance" if acc < 0.55 else "⚠ above chance — investigate"
        elif condition == "morgan_delta_opp":
            flag = "oracle (not deployable)"

        print(f"{target:<15} {condition:<28} {auroc:>7.4f} {mcc:>7.4f} {acc:>9.4f}  {flag}")
    print()

Target          Condition                      AUROC     MCC  Accuracy Check
--------------------------------------------------------------------------------
R/S_class       morgan_count_chiral           0.9442  0.6930    0.8463  
R/S_class       morgan_count_no_chiral        0.5000  0.0000    0.5000  ✓ at chance
R/S_class       morgan_delta_opp              0.9366  0.6882    0.8411  oracle (not deployable)
R/S_class       morgan_delta_nos              0.9240  0.6470    0.8211  
R/S_class       morgan_cgr                    0.9309  0.6858    0.8421  

@/@@_class      morgan_count_chiral           0.8029  0.4200    0.7095  
@/@@_class      morgan_count_no_chiral        0.5000  0.0000    0.5000  ✓ at chance
@/@@_class      morgan_delta_opp              0.7735  0.3737    0.6853  oracle (not deployable)
@/@@_class      morgan_delta_nos              0.7589  0.3352    0.6663  
@/@@_class      morgan_cgr                    0.7786  0.3821    0.6905  

F/L_class       morgan_count_chiral       

# Summary of Results: Featurization Analysis

The evaluation of Random Forest performance across three targets of increasing chemical difficulty (`R/S`, `@/@@`, `F/L`) reveals the following:

1. Baseline Superiority: **The standard `morgan_count_chiral` fingerprint is the most robust representation.** It achieves the highest AUROC across all three prediction targets, outperforming both the `delta_opp` "oracle" and the CGR-style factorization.

2. Deployability: Unlike the `morgan_delta_opp` condition, which provides an artificial antisymmetric signal that cannot be generated for single, unpaired query molecules, `morgan_count_chiral` is fully deployable using only the query molecule's SMILES.

3. Factorization vs. Interleaving: While `morgan_cgr` and `morgan_delta_nos` attempt to explicitly factorize the achiral scaffold from the chiral signal, the data shows that the standard 512-bit chiral Morgan fingerprint already implicitly contains the necessary information for a non-linear model to decode complex physical elution orders. The explicit factorization did not yield performance gains.

Recommendation: For real-world applications in chiral elution order prediction, use `morgan_count_chiral`. It provides the best performance/complexity balance, avoids the distributional mismatch of paired-delta methods, and leverages RDKit’s built-in stereochemical hashing to capture the 3D-dependent features required for chromatography tasks.

### More Detailed Summary of Results:
Negative control worked as expected.
- All metrics show that the **RF model is incapable of any predictive power beyond random chance when `includeChirality` is set to False** for the fingerprint. A chirality-blind Morgan fingerprint produces a representation where enantiomers are literally identical vectors
- Note: F1 shows a tiny non-zero std (±0.021–0.036) across all three targets. F1 is sensitive to the threshold and to which class the model arbitrarily assigns when features are tied, so minor implementation-level randomness can produce small fluctuations even when the model has zero discriminative power.
  
**Results show that predicting R/S is the easiest target, which is expected.**
- Predicting @/@@ is a genuinely difficult prediction task because the label is a SMILES artifact that doesn't align well with what Morgan fingerprints encode.
- **F/L is the most important scientific target and the AUROC of around 0.86 indicates that Morgan count fingerprints carry meaningful signal.**

**Important differences relative to Table 1 from the paper**
Table 1 from the paper [link](https://link.springer.com/article/10.1186/s13321-025-01080-7) reports a test set accuracy of 0.912 for predicting CIP R/S label. When using different splits in this notebook, the results above show an average accuracy of only 0.84. Potential explanations include:
- Their code does use different splits (and [only 1 fold](https://github.com/jairesdesousa/chiraldlsv/blob/main/RF_models/RF_models.py#L27))
- Their code allowed `max_depth` equal the default value of `None` [link](https://github.com/jairesdesousa/chiraldlsv/blob/main/RF_models/RF_models.py#L62) while I chose to regularize `max_depth=25` here
- Their code used `random_state=0` while I used `random_state=42`

Table 2 from the paper shows a test set accuracy of 0.716 for predicting the SMILES stereochemical label of @ vs. @@. which is roughly inline with the 0.70 value shown above.

# Save results

**`3_RF_morgan_count_results.csv`** — all splits (train, val, test), 225 rows. Save results on all splits. But be sure to filter to `split == "test"` (75 rows: 5 conditions × 3 targets × 5 folds) when running Tukey HSD in the final analysis notebook.

In [20]:
# Save full results (train + val + test) for records
# Tukey HSD will only use test-set results — filter on split == "test"
output_path = "3_RF_morgan_count_results.csv"
all_results_df.to_csv(output_path, index=False)
print(f"Saved {len(all_results_df)} rows to {output_path}")
print(f"  Conditions: {all_results_df['featurization'].unique().tolist()}")
print(f"  Targets:    {all_results_df['target'].unique().tolist()}")
print(f"  Folds:      {sorted(all_results_df['fold'].unique().tolist())}")
print(f"  Splits:     {sorted(all_results_df['split'].unique().tolist())}")
print(f"  Columns:    {all_results_df.columns.tolist()}")

Saved 225 rows to 3_RF_morgan_count_results.csv
  Conditions: ['morgan_count_chiral', 'morgan_count_no_chiral', 'morgan_delta_opp', 'morgan_delta_nos', 'morgan_cgr']
  Targets:    ['R/S_class', '@/@@_class', 'F/L_class']
  Folds:      [0, 1, 2, 3, 4]
  Splits:     ['test', 'train', 'val']
  Columns:    ['AUROC', 'MCC', 'Accuracy', 'F1', 'AUPRC', 'fold', 'model', 'featurization', 'target', 'split']
